## 11. Comparacion de Modelos — Regresion Logistica, Poisson, GAM y Random Forest

Compara cuatro familias de modelos contra XGBoost para el problema de prediccion de dengue grave binario.
Todos los experimentos se registran en MLflow bajo el mismo experimento `dengue-grave-xgboost`.

| Modelo | Tipo | Target |
|---|---|---|
| Regresion Logistica | Clasificacion binaria (GLM logit) | grave_bin |
| Regresion de Poisson | GLM Poisson (cuenta) | grave (count) |
| GAM de Poisson | GAM con familia Poisson | grave (count) |
| Random Forest | Ensamble de arboles | grave_bin |
| XGBoost | Gradient boosting | grave_bin |

Para Poisson y GAM, la prediccion es el conteo esperado; se binariza con umbral 0.5 para evaluar clasificacion.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    roc_curve, auc, precision_recall_curve
)
import xgboost as xgb

# MLflow — cambiar URI a EC2 cuando este disponible
MLFLOW_TRACKING_URI = 'http://127.0.0.1:5000'
EXPERIMENT_NAME     = 'dengue-grave-xgboost'

DATA_PATH = '../data/processed/dengue_features_modelado.csv'
FIG_DIR   = '../data/figures'

FEATURE_COLS = (
    [f'grave_lag_{l}'   for l in [1,2,3,4,6]] +
    [f'clasico_lag_{l}' for l in [1,2,3,4,6]] +
    ['grave_roll3', 'clasico_roll3'] +
    ['temp_mean_c', 'temp_lag_1', 'temp_lag_2', 'temp_lag_3'] +
    ['rain_mm_day', 'rain_lag_1', 'rain_lag_2', 'rain_lag_3'] +
    ['mes_sin', 'mes_cos', 'anio_epidemia', 'ANO', 'MES'] +
    ['es_endemico', 'zona_canal_lag1', 'p25', 'p75', 'sir_lag1']
)  # 30 features mensuales

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f'Features ({len(FEATURE_COLS)}): listo.')

### 1. Datos y particion

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df['grave_bin'] = (df['grave'] > 0).astype(int)

train = df[df['ANO'] <= 2021]
val   = df[df['ANO'] == 2022]
test  = df[df['ANO'] >= 2023]

X_train_raw = train[FEATURE_COLS].fillna(0)
X_val_raw   = val[FEATURE_COLS].fillna(0)
X_test_raw  = test[FEATURE_COLS].fillna(0)
y_train_bin = train['grave_bin']
y_val_bin   = val['grave_bin']
y_test_bin  = test['grave_bin']
y_train_cnt = train['grave']          # Para modelos de conteo
y_val_cnt   = val['grave']
y_test_cnt  = test['grave']

# Escalar para LR y Poisson
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_raw)
X_val_sc   = scaler.transform(X_val_raw)
X_test_sc  = scaler.transform(X_test_raw)

spw = (y_train_bin == 0).sum() / (y_train_bin == 1).sum()
print(f'Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')
print(f'scale_pos_weight: {spw:.1f}')

### 2. Funciones de evaluacion

In [ ]:
def metricas(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    return {
        'auroc':         float(roc_auc_score(y_true, y_prob)),
        'avg_precision': float(average_precision_score(y_true, y_prob)),
        'f1':            float(f1_score(y_true, y_pred, zero_division=0)),
        'recall':        float(recall_score(y_true, y_pred, zero_division=0)),
        'precision':     float(precision_score(y_true, y_pred, zero_division=0)),
    }

def best_thr(y_val, prob_val):
    thrs = np.arange(0.05, 0.95, 0.01)
    f1s  = [f1_score(y_val, (prob_val>=t).astype(int), zero_division=0) for t in thrs]
    return float(thrs[int(np.argmax(f1s))])

resultados = {}  # nombre -> dict de metricas en test

### 3. Regresion Logistica

GLM con enlace logit. Es el modelo discriminativo lineal de referencia para clasificacion binaria. Util para verificar que XGBoost aporta valor mas alla de una separacion lineal.

In [ ]:
with mlflow.start_run(run_name='logistic-regression'):
    lr = LogisticRegression(
        C=0.1,              # regularizacion L2
        class_weight='balanced',  # equivalente a scale_pos_weight
        max_iter=500,
        n_jobs=-1,
        random_state=42,
    )
    lr.fit(X_train_sc, y_train_bin)
    mlflow.log_params({'model':'LogisticRegression','C':0.1,'class_weight':'balanced'})

    prob_val_lr  = lr.predict_proba(X_val_sc)[:, 1]
    prob_test_lr = lr.predict_proba(X_test_sc)[:, 1]

    thr_lr = best_thr(y_val_bin, prob_val_lr)
    m_val  = metricas(y_val_bin,  prob_val_lr,  thr_lr)
    m_test = metricas(y_test_bin, prob_test_lr, thr_lr)

    mlflow.log_metrics({f'val_{k}': v for k, v in m_val.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in m_test.items()})
    mlflow.log_param('best_threshold', round(thr_lr, 2))
    mlflow.sklearn.log_model(lr, 'model')

    resultados['Logistica'] = {'val': m_val, 'test': m_test}
    print(f'LR val  AUROC={m_val["auroc"]:.3f}  AP={m_val["avg_precision"]:.3f}  F1={m_val["f1"]:.3f}')
    print(f'LR test AUROC={m_test["auroc"]:.3f}  AP={m_test["avg_precision"]:.3f}  F1={m_test["f1"]:.3f}')

### 4. Regresion de Poisson

Modela el conteo esperado de casos graves: E[grave] = exp(Xb). Es el modelo parametrico natural para datos de conteo epidemiologicos. Para evaluacion de clasificacion, binarizamos la prediccion en prob > umbral, donde la 'probabilidad' se deriva de la distribucion de Poisson: P(Y>0) = 1 - exp(-lambda).

In [ ]:
with mlflow.start_run(run_name='poisson-regression'):
    poisson = PoissonRegressor(
        alpha=0.1,   # regularizacion L2
        max_iter=300,
    )
    poisson.fit(X_train_sc, y_train_cnt)
    mlflow.log_params({'model':'PoissonRegressor','alpha':0.1})

    # lambda = conteo predicho; P(Y>0 | lambda) = 1 - exp(-lambda)
    lambda_val  = np.clip(poisson.predict(X_val_sc),  0, None)
    lambda_test = np.clip(poisson.predict(X_test_sc), 0, None)
    prob_val_po  = 1 - np.exp(-lambda_val)
    prob_test_po = 1 - np.exp(-lambda_test)

    thr_po = best_thr(y_val_bin, prob_val_po)
    m_val  = metricas(y_val_bin,  prob_val_po,  thr_po)
    m_test = metricas(y_test_bin, prob_test_po, thr_po)

    mlflow.log_metrics({f'val_{k}': v for k, v in m_val.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in m_test.items()})
    mlflow.log_param('best_threshold', round(thr_po, 2))
    mlflow.sklearn.log_model(poisson, 'model')

    resultados['Poisson'] = {'val': m_val, 'test': m_test}
    print(f'Poisson val  AUROC={m_val["auroc"]:.3f}  AP={m_val["avg_precision"]:.3f}  F1={m_val["f1"]:.3f}')
    print(f'Poisson test AUROC={m_test["auroc"]:.3f}  AP={m_test["avg_precision"]:.3f}  F1={m_test["f1"]:.3f}')

### 5. GAM de Poisson

Modelo Aditivo Generalizado con familia Poisson. Extiende la regresion de Poisson
permitiendo relaciones no lineales (splines) entre cada feature y la respuesta.
Mas interpretable que XGBoost: se pueden graficar las funciones parciales fi(xi)
para comunicar resultados a epidemiologos sin conocimientos de ML.

In [ ]:
from pygam import PoissonGAM, s
from sklearn.utils import resample

GAM_FEATURES = [
    'grave_lag_1', 'grave_lag_2', 'grave_lag_4', 'grave_roll3',
    'clasico_lag_1', 'clasico_roll3',
    'sir_lag1', 'zona_canal_lag1', 'p75',
    'mes_sin', 'mes_cos', 'ANO'
]

# GAM es O(n^2) en memoria con splines — muestra estratificada de 30k para entrenar
train_sample = resample(train, n_samples=30_000, random_state=42, stratify=train['grave_bin'])
X_tr_gam = train_sample[GAM_FEATURES].fillna(0).values
y_tr_gam  = train_sample['grave'].values
X_va_gam  = val[GAM_FEATURES].fillna(0).values
X_te_gam  = test[GAM_FEATURES].fillna(0).values

with mlflow.start_run(run_name='gam-poisson'):
    gam = PoissonGAM(
        s(0)+s(1)+s(2)+s(3)+s(4)+s(5)+s(6)+s(7)+s(8)+s(9)+s(10)+s(11)
    ).fit(X_tr_gam, y_tr_gam)

    mlflow.log_params({
        'model': 'PoissonGAM',
        'gam_backend': 'pygam',
        'n_train_sample': 30_000,
        'n_features_gam': len(GAM_FEATURES),
        'features_gam': ','.join(GAM_FEATURES),
    })

    lambda_val_gam  = np.clip(gam.predict(X_va_gam), 0, None)
    lambda_test_gam = np.clip(gam.predict(X_te_gam), 0, None)
    # P(Y > 0 | lambda) = 1 - exp(-lambda)
    prob_val_gam  = 1 - np.exp(-lambda_val_gam)
    prob_test_gam = 1 - np.exp(-lambda_test_gam)

    thr_gam = best_thr(y_val_bin, prob_val_gam)
    m_val   = metricas(y_val_bin,  prob_val_gam,  thr_gam)
    m_test  = metricas(y_test_bin, prob_test_gam, thr_gam)

    mlflow.log_metrics({f'val_{k}':  v for k, v in m_val.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in m_test.items()})
    mlflow.log_param('best_threshold', round(thr_gam, 2))

    resultados['GAM-Poisson'] = {'val': m_val, 'test': m_test}
    print(f'GAM  val  AUROC={m_val["auroc"]:.3f}  AP={m_val["avg_precision"]:.3f}  F1={m_val["f1"]:.3f}')
    print(f'GAM  test AUROC={m_test["auroc"]:.3f}  AP={m_test["avg_precision"]:.3f}  F1={m_test["f1"]:.3f}')

### 6. Random Forest

Ensamble de arboles de decision. Sirve como comparacion directa con XGBoost: ambos son modelos basados en arboles, pero RF usa bagging mientras XGBoost usa boosting. Se espera que XGBoost supere a RF, pero RF es util para validar que el boosting agrega valor.

In [ ]:
with mlflow.start_run(run_name='random-forest'):
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42,
    )
    rf.fit(X_train_raw, y_train_bin)
    mlflow.log_params({
        'model': 'RandomForest',
        'n_estimators': 200,
        'max_depth': 10,
        'class_weight': 'balanced'
    })

    prob_val_rf  = rf.predict_proba(X_val_raw)[:, 1]
    prob_test_rf = rf.predict_proba(X_test_raw)[:, 1]

    thr_rf = best_thr(y_val_bin, prob_val_rf)
    m_val  = metricas(y_val_bin,  prob_val_rf,  thr_rf)
    m_test = metricas(y_test_bin, prob_test_rf, thr_rf)

    mlflow.log_metrics({f'val_{k}': v for k, v in m_val.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in m_test.items()})
    mlflow.log_param('best_threshold', round(thr_rf, 2))
    mlflow.sklearn.log_model(rf, 'model')

    resultados['RandomForest'] = {'val': m_val, 'test': m_test}
    print(f'RF   val  AUROC={m_val["auroc"]:.3f}  AP={m_val["avg_precision"]:.3f}  F1={m_val["f1"]:.3f}')
    print(f'RF   test AUROC={m_test["auroc"]:.3f}  AP={m_test["avg_precision"]:.3f}  F1={m_test["f1"]:.3f}')

### 7. XGBoost (referencia)

Re-ejecuta XGBoost v1 del notebook 10 para tener todas las metricas en el mismo run de MLflow.

In [ ]:
with mlflow.start_run(run_name='xgboost-v1-ref'):
    spw = float((y_train_bin == 0).sum() / (y_train_bin == 1).sum())
    xgb_model = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw, eval_metric='aucpr',
        early_stopping_rounds=30, random_state=42,
        n_jobs=-1, tree_method='hist'
    )
    xgb_model.fit(X_train_raw, y_train_bin,
                  eval_set=[(X_val_raw, y_val_bin)], verbose=False)
    mlflow.log_params({'model':'XGBoost','scale_pos_weight':round(spw,1),
                       'best_iteration':xgb_model.best_iteration})

    prob_val_xgb  = xgb_model.predict_proba(X_val_raw)[:, 1]
    prob_test_xgb = xgb_model.predict_proba(X_test_raw)[:, 1]

    thr_xgb = best_thr(y_val_bin, prob_val_xgb)
    m_val   = metricas(y_val_bin,  prob_val_xgb,  thr_xgb)
    m_test  = metricas(y_test_bin, prob_test_xgb, thr_xgb)

    mlflow.log_metrics({f'val_{k}': v for k, v in m_val.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in m_test.items()})
    mlflow.xgboost.log_model(xgb_model, 'model')

    resultados['XGBoost'] = {'val': m_val, 'test': m_test}
    print(f'XGB  val  AUROC={m_val["auroc"]:.3f}  AP={m_val["avg_precision"]:.3f}  F1={m_val["f1"]:.3f}')
    print(f'XGB  test AUROC={m_test["auroc"]:.3f}  AP={m_test["avg_precision"]:.3f}  F1={m_test["f1"]:.3f}')

### 8. Tabla comparativa

In [ ]:
rows = []
for nombre, res in resultados.items():
    for split, m in res.items():
        rows.append({
            'Modelo': nombre, 'Conjunto': split,
            'AUROC': round(m['auroc'], 3),
            'Avg Precision': round(m['avg_precision'], 3),
            'F1': round(m['f1'], 3),
            'Recall': round(m['recall'], 3),
            'Precision': round(m['precision'], 3),
        })

comp = pd.DataFrame(rows)
print(comp.to_string(index=False))
comp.to_csv('../data/processed/comparacion_modelos.csv', index=False)
print('\nGuardado: data/processed/comparacion_modelos.csv')

### 9. Curvas ROC comparativas (validacion 2022)

In [ ]:
probs_val = {
    'Logistica':   prob_val_lr,
    'Poisson':     prob_val_po,
    'GAM-Poisson': prob_val_gam,
    'RandomForest':prob_val_rf,
    'XGBoost':     prob_val_xgb,
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['tab:blue','tab:orange','tab:green','tab:red','tab:purple']

for (nombre, prob), color in zip(probs_val.items(), colors):
    fpr, tpr, _ = roc_curve(y_val_bin, prob)
    prec, rec, _ = precision_recall_curve(y_val_bin, prob)
    ap  = average_precision_score(y_val_bin, prob)
    au  = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, label=f'{nombre} (AUROC={au:.3f})')
    axes[1].plot(rec, prec, color=color, label=f'{nombre} (AP={ap:.3f})')

axes[0].plot([0,1],[0,1],'k:', label='Aleatorio')
axes[0].set(xlabel='FPR', ylabel='TPR', title='Curvas ROC - Validacion 2022')
axes[0].legend(fontsize=8)

axes[1].set(xlabel='Recall', ylabel='Precision',
            title='Curvas Precision-Recall - Validacion 2022')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/figures/15_comparacion_modelos.png', dpi=120)
plt.close()
print('Figura guardada: 15_comparacion_modelos.png')

### 10. Conclusion de la comparacion

Resultados reales (val = 2022, test = 2023-2024):

| Modelo | Val AUROC | Val AP | Val F1 | Test AUROC | Test AP | Test F1 |
|---|---|---|---|---|---|---|
| Regresion Logistica | 0.758 | 0.236 | 0.329 | 0.706 | 0.195 | 0.262 |
| Regresion de Poisson | 0.745 | 0.205 | 0.322 | 0.683 | 0.192 | 0.256 |
| GAM de Poisson | 0.764 | 0.232 | 0.345 | 0.683 | 0.186 | 0.193 |
| Random Forest | 0.781 | 0.287 | 0.346 | 0.733 | 0.211 | 0.288 |
| **XGBoost** | **0.787** | **0.304** | **0.350** | **0.728** | **0.202** | **0.288** |

**Hallazgos:**
- **XGBoost** tiene el mayor AP en validacion (0.304) y el mejor F1, confirmando que el boosting captura interacciones no lineales que los modelos lineales no ven.
- **Random Forest** es el segundo modelo mas fuerte (val AUROC 0.781 vs 0.787), con diferencia marginal. La brecha de AP (0.287 vs 0.304) justifica XGBoost para el sistema de produccion.
- **GAM de Poisson** logra val AUROC 0.764, por encima de Logistica (0.758) y Poisson GLM (0.745), validando el aporte de los splines. Su ventaja es la interpretabilidad: cada fi(xi) se puede graficar para comunicar el efecto de cada variable a epidemiologos.
- **Regresion de Poisson** (GLM) es el modelo de referencia interpretable; su AUROC de 0.745 en val confirma senial lineal explotable en los datos.
- La caida de test vs val en todos los modelos (~0.05 AUROC) sugiere drift temporal entre 2022 y 2023-2024, no sobreajuste especifico de XGBoost.

**Modelo seleccionado para produccion:** XGBoost (mejor AP y F1 en validacion). GAM de Poisson queda como modelo secundario para contextos de comunicacion con autoridades sanitarias.